# Week 5 · Day 13 — Capstone Build — Session 1: Pipeline + Model + UI

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit Supervisor, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L13 — Capstone: Build

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay
**Module:** 3 · **Week:** 5 · **Tier:** Intro
**New concept:** (Capstone build session 1 — no new concept today)
**Deliverable wired in:** D3 preparation — Capstone Notebook + Project Brief (40%)

## Learning objectives
By the end of today you will have a **first working version** of your capstone with three parts:
1. a **data pipeline** (load + clean),
2. a **model or API integration**, and
3. an **initial UI**.

## Why this matters

This is where the whole course comes together for **your** problem — the one you chose in Deliverable 1. Below is a complete, runnable reference (TIA Lite: the delay-risk classifier behind a small app). Treat it as a **template**: keep the structure, and swap in your own dataset, model, and labels at the points marked `# TODO`.

## Environment setup

In [ ]:
# Gradio for the UI; google-genai if your capstone uses an LLM.
!pip install -q gradio google-genai
print("Environment ready.")

In [2]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

Model configured: gemini-2.0-flash
Route R12 (Wilberforce → CBD). The 07:45 service, operated by OP-104 on vehicle SLPTA-1142, departed 25 minutes late. Recorded cause: Heavy traffic on Wilkinson Road. About 40 passengers were affected and the dispatch desk received multiple complaints. (Synthetic SLPTA scenario — no real data.)


## Concept — the capstone pipeline

Every capstone, whatever the problem, follows the same shape: **raw data → clean → model or API → a UI a user can open.** Build it one block at a time and test each block before moving on.

<p align="center"></p>

## Choose your path
- **Path A — classical ML** (no API key): predict something from `route_logs` or `complaints` (delay risk, complaint category, etc.). *The reference below uses this path.*
- **Path B — LLM** (needs `GEMINI_API_KEY`): summarise / classify / extract from `complaints`, `policies`, or `incident_reports`.

Keep the three numbered steps; change what is inside them for your problem.

## Assessment Rubric — D3 Capstone Notebook + Project Brief

Your capstone is assessed on four dimensions. **Pass threshold: Proficient (2) or above on every dimension.**

| Dimension | Emerging (1) | Proficient (2) | Strong (3) | Exemplary (4) |
|-----------|-------------|----------------|------------|---------------|
| **Correctness** | Code runs with errors; model output is wrong | Code runs end-to-end; output addresses the SLPTA problem | Edge cases handled; output is consistently accurate | Robust pipeline with validation; output exceeds expectations |
| **Grounding** | No data used; model hallucinates freely | Model answer is grounded in Route R12 data or policy doc | Retrieval is accurate; sources cited in output | Multi-source grounding; uncertainty flagged when data is missing |
| **Clarity** | Notebook is hard to follow; no narrative | Notebook tells a clear story from problem to solution | Each section has a purpose; non-technical reader could follow | Notebook could be shown to SLPTA management as-is |
| **Safety** | No consideration of failure modes | At least one failure mode identified and mitigated | Hallucination and bias risks documented | Ethics checklist integrated; refusal pattern demonstrated |

### Step 1 — Data pipeline

<!-- cell-diagram:c11 -->
<p align="center"></p>

### Check your understanding (before running)
Capstone step 1: load and clean your dataset (template uses route logs).

**Predict:** After cleaning, will row count drop because of duplicates or missing drops?

In [3]:
import pandas as pd

# --- STEP 1: DATA LOADING ---
# Load the raw transportation logs and remove any duplicate entries to ensure data integrity.
df = load_route_logs().drop_duplicates()

# --- STEP 2: DATA CLEANING ---
# Filter out extreme outliers (delays over 3 hours) which might represent data entry errors or non-standard incidents.
df = df[df["delay_minutes"] < 180].copy()

# Handle missing values for 'weather' by imputing the most frequent value (mode).
df["weather"] = df["weather"].fillna(df["weather"].mode()[0])

# Handle missing values for 'passenger_count' by imputing the median value to minimize the impact of outliers.
df["passenger_count"] = df["passenger_count"].fillna(df["passenger_count"].median())

# --- STEP 3: TARGET & FEATURE DEFINITION ---
# Define the target variable: 1 if the delay is greater than 15 minutes (High Risk), else 0.
target = (df["delay_minutes"] > 15).astype(int)

# Select the features that will be used for prediction.
feature_cols = ["route_id", "peak_period", "weather", "cause_category",
                "scheduled_hour", "distance_km", "passenger_count"]

# --- STEP 4: PRE-PROCESSING ---
# Convert categorical string variables into dummy/indicator variables (One-Hot Encoding).
X = pd.get_dummies(df[feature_cols])

# Summary output to verify the pipeline status.
print(f"Pipeline ready: {X.shape[0]} rows, {X.shape[1]} features.")

Pipeline ready: 1853 rows, 19 features.


### Check your understanding (before moving on)
In one sentence: what does your data pipeline do?
Describe it as if explaining to a Route R12 dispatcher who has never seen code.

*Write your answer below, then continue to Step 2.*

### Step 2 — Model (or API integration)

<!-- cell-diagram:c14 -->
<p align="center"></p>

### Check your understanding (before running)
Capstone: split features into train/test before fitting your model.

**Predict:** Will your test accuracy stay close to train if the model is not overfitting?

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score
import numpy as np

# --- STEP 1: DATA SPLITTING ---
# Split the dataset into training (75%) and testing (25%) sets.
# 'stratify=target' ensures both sets have the same proportion of delay/no-delay cases.
Xtr, Xte, ytr, yte = train_test_split(X, target, test_size=0.25,
                                      random_state=0, stratify=target)

# --- STEP 2: MODEL TRAINING ---
# Initialize a Decision Tree Classifier with a max_depth of 5 to prevent overfitting.
# We fit the model using the training features (Xtr) and training labels (ytr).
model = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xtr, ytr)

# --- STEP 3: EVALUATION ---
# Calculate and print performance metrics to understand how well the model predicts on unseen data.
predictions = model.predict(Xte)
print(f"Accuracy: {accuracy_score(yte, predictions):.1%}")
print(f"Recall: {recall_score(yte, predictions):.1%}")

# --- STEP 4: PREDICTION FUNCTION FOR UI ---
def predict_delay_risk(route_id, peak, weather, cause, hour, distance, passengers):
    """
    Takes raw inputs from the UI, converts them to the format the model expects,
    and returns a user-friendly recommendation.
    """
    # Create a single-row DataFrame from the input arguments
    row = pd.DataFrame([{
        "route_id": route_id,
        "peak_period": peak,
        "weather": weather,
        "cause_category": cause,
        "scheduled_hour": hour,
        "distance_km": distance,
        "passenger_count": passengers
    }])

    # Apply One-Hot Encoding and align columns with the training set (X.columns)
    # .reindex ensures the new row has all columns used during training, filling missing ones with 0.
    row = pd.get_dummies(row).reindex(columns=X.columns, fill_value=0)

    # Get the probability of the 'High Risk' class (index 1)
    p = model.predict_proba(row)[0][1]

    # Return a formatted string for the Gradio UI
    status = "PRE-WARN passengers" if p > 0.5 else "looks on time"
    return f"Badly-delayed risk: {p:.0%}  ->  {status}"

ModuleNotFoundError: No module named 'sklearn'

### Step 3 — Initial UI

*(Path B users: make the function call `ask(...)` instead, and use simple text in/out.)*

<!-- cell-diagram:c16 -->
<p align="center"></p>

### Check your understanding (before running)
Capstone: launch Gradio so dispatch can try your classifier live.

**Predict:** What label will the app show for peak hour + heavy rain on **R12**?

In [7]:
import gradio as gr

# --- STEP 1: UI DEFINITION ---
# We use gr.Interface to map our prediction function to UI components.
demo = gr.Interface(
    fn=predict_delay_risk,  # The function defined in the previous step
    inputs=[
        # Dropdown for selecting the specific route ID
        gr.Dropdown(["R12","R7","R3","R21","R9","R14"], label="Route", value="R12"),
        # Checkbox for binary Peak/Off-Peak status
        gr.Checkbox(label="Peak period?"),
        # Dropdown for weather conditions known to affect transit times
        gr.Dropdown(["Sunny","Rain","Heavy Rain","Harmattan"], label="Weather", value="Heavy Rain"),
        # Dropdown for the primary cause of potential delays
        gr.Dropdown(["Normal","Traffic","Mechanical","Weather","Staffing"], label="Cause", value="Traffic"),
        # Slider for the hour of the day (24h format)
        gr.Slider(6, 20, value=8, step=1, label="Scheduled hour"),
        # Number input for distance; defaults to a standard Route 12 length
        gr.Number(value=11.2, label="Distance (km)"),
        # Number input for the current or expected passenger load
        gr.Number(value=40, label="Passengers"),
    ],
    outputs=gr.Textbox(label="Dispatch advice"),
    title="Transport Intelligent Assistant Lite — Delay-risk early warning",
    description="Reference capstone. Replace inputs/outputs with your own problem.",
)

# --- STEP 2: LAUNCH ---
# This generates a public link and displays the UI directly in the notebook.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d35f5e4594707c883.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Facilitator check-in (end of session)
Show your facilitator: (1) data pipeline runs, (2) model trains or API call works, (3) UI launches. Note one thing you will fix tomorrow.

> **If you remember one thing from today's build session:**
> Every AI project is data + model + UI, built one block at a time.
> You have the data. Today you wired the model. Tomorrow you add the UI.

## Submission checklist
- [ ] Step 1 data pipeline runs without error
- [ ] Step 2 model trains (or API call returns) for **your** problem
- [ ] Step 3 UI launches and returns a result
- [ ] One-line note on what to improve in Session 2
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)